# 6 · Relation extraction with Gemini  (per article)

Open information extraction of the **relations** that hold between the entities we already have.
This is the relation-side twin of `2_oie_extraction.ipynb` (which extracts entities + types): here
Gemini is handed the article's **already-typed, deduplicated entities** (from nb4) and asked to state
every relation between them as a `[subject, relation_phrase, object]` triple with a verbatim evidence
span — inventing the free-form `relation_phrase` that best fits, exactly as nb2 invents type labels.

Reads the **final entity nodes** for the article — `typed_deduped_entities_ext_consolidated.json` if
present, else `typed_deduped_entities_ext.json` (both from nb4). Each entity node is given a stable
id (`E1`, `E2`, ...) so the model can point at subjects/objects unambiguously and nb7 can link them
back. Writes `results/<ARTICLE_ID>/oie_relations.json`.

Set `ARTICLE_ID` for the single-article demo cell, or list ids in `SELECTED_ARTICLES` and run the
**batch** cell. *(Canonicalizing each `relation_phrase` to a schema.org / `ex:` property is the next
stage — `7_property_extension.ipynb`.)*

In [16]:
# --- Config -------------------------------------------------------------------------------
from pathlib import Path

RESULTS_DIR  = Path("results")            # reads results/<id>/typed_deduped_entities_ext*.json (nb4)
DATASET_DIR  = Path("dataset/articles")   # reads <id>.txt for the REFERENCE_DATE and relation evidence
GEMINI_MODEL = "gemini-2.5-flash"

ARTICLE_ID   = "art1"                      # single article used by the demo cell below

# Which articles the BATCH cell processes, in one run.
#   None -> every article that has an entity file (typed_deduped_entities_ext*.json), sorted.
#   list -> exactly these ids, in this order, e.g. ["art1", "art2", "art20"].
SELECTED_ARTICLES = ["art1", "art2", "art4", "art5"]

# Which per-article entity file to use as the relation arguments. The consolidated file (post global
# re-consolidation in nb4) is preferred when present; otherwise the raw incremental one.
ENTITY_FILES = ["typed_deduped_entities_ext_consolidated.json", "typed_deduped_entities_ext.json"]

def entity_path(aid):
    for name in ENTITY_FILES:
        p = RESULTS_DIR / aid / name
        if p.exists():
            return p
    return None

_available = sorted(p.parent.name for p in RESULTS_DIR.glob("*/typed_deduped_entities_ext*.json"))
_available = sorted(set(_available))
if SELECTED_ARTICLES is None:
    ARTICLE_IDS = _available
else:
    missing = [a for a in SELECTED_ARTICLES if a not in _available]
    if missing:
        print("WARNING: skipping (no entity file from nb4):", missing)
    ARTICLE_IDS = [a for a in SELECTED_ARTICLES if a in _available]
assert ARTICLE_IDS, "No runnable articles (need nb4 typed_deduped_entities_ext*.json). Check SELECTED_ARTICLES."
print("article (demo):", ARTICLE_ID, "| batch:", ARTICLE_IDS, "| results:", RESULTS_DIR)

article (demo): art1 | batch: ['art1', 'art2', 'art4', 'art5'] | results: results


In [3]:
# --- Gemini client -----------------------------------------------------------------------
%pip install -q google-genai

import os, json, time, re
from google import genai
from google.genai import types as genai_types
from IPython.display import Markdown, display

GEMINI_API_KEY = "" or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
assert GEMINI_API_KEY, "Paste your key in GEMINI_API_KEY above, or set it in the env and restart the kernel."

client = genai.Client(api_key=GEMINI_API_KEY)
last_latency_s = None

def gemini_json(system_prompt, user_message, model=None, temperature=0):
    """Call Gemini with JSON-mode output; return the parsed object and record latency."""
    global last_latency_s
    t0 = time.perf_counter()
    resp = client.models.generate_content(
        model=model or GEMINI_MODEL,
        contents=user_message,
        config=genai_types.GenerateContentConfig(
            system_instruction=system_prompt,
            temperature=temperature,
            response_mime_type="application/json",
        ),
    )
    last_latency_s = time.perf_counter() - t0
    return json.loads(resp.text)

Note: you may need to restart the kernel to use updated packages.


## 1. Load the article's entities (relation arguments)

Every entity node from nb4 becomes one **entity card** — a stable id, its canonical surface text,
its chosen type, and its aliases — that the model may reference as a subject or object. The id map is
persisted alongside the relations so nb7 can resolve `subject_id` / `object_id` back to entity nodes.

In [4]:
def load_entities(aid):
    """Read the article's final entity nodes (nb4) and build the id-stamped entity cards.
    Returns (entities_file_basename, cards) where each card is {id, text, type, aliases}."""
    p = entity_path(aid)
    assert p is not None, f"no entity file for {aid} (expected one of {ENTITY_FILES} under results/{aid}/)"
    data = json.loads(p.read_text())
    cards = []
    for i, e in enumerate(data.get("entities", []), 1):
        cards.append({
            "id":      f"E{i}",
            "text":    e.get("canonical_text"),
            "type":    e.get("chosen_type"),
            "aliases": e.get("aliases", []),
        })
    return p.name, cards

def _reference_date(text):
    """Parse the 'Date: DD/MM/YYYY' dateline from the article header -> ISO-8601 (YYYY-MM-DD), or None."""
    m = re.search(r"^Date:\s*(\d{2})/(\d{2})/(\d{4})", text, re.MULTILINE)
    if not m:
        return None
    d, mth, y = m.groups()
    return f"{y}-{mth}-{d}"

# quick look at what the demo article's arguments are
_fname, _cards = load_entities(ARTICLE_ID)
print(f"{ARTICLE_ID}: {len(_cards)} entities from {_fname}")
for c in _cards[:8]:
    print(f"  {c['id']:<4} {c['text']!r:<28} :: {c['type']}")

art1: 103 entities from typed_deduped_entities_ext_consolidated.json
  E1   'France'                     :: Country
  E2   'the UK'                     :: Country
  E3   'Portugal'                   :: Country
  E4   'Spain'                      :: Country
  E5   'Belgium'                    :: Country
  E6   'the Netherlands'            :: Country
  E7   'the country'                :: Country
  E8   '2,025 excess deaths'        :: QuantitativeValue


## 2. Open relation extraction

In [5]:
# Free-form relation extraction over the given entities -> results/<id>/oie_relations.json
REL_OIE_SYSTEM_PROMPT = """You are an expert Open Relation Extraction system.
You are given a TEXT to analyze, a REFERENCE_DATE (the article's publication date, ISO-8601) for context, and a list of ENTITIES already extracted from that TEXT. Each entity has an id, its surface text, its type, and optional aliases.
TASK: Extract every relation that the TEXT explicitly states between two of the given ENTITIES. Represent each relation as a directed triple [subject, relation_phrase, object], where subject and object are two DIFFERENT entities from the ENTITIES list. For the relation_phrase, choose freely the label that best captures the relation — you are not given a fixed set of relations: invent whatever concise phrase fits. After extraction, produce a definition for every unique relation_phrase you used.
RULES:
1. Subject and object MUST both come from the given ENTITIES list — reference them by their exact id (e.g. "E7"). Never introduce an entity that is not in the list, and never relate an entity to itself.
2. Direction matters: the triple reads subject --relation_phrase--> object. Orient every triple so it reads correctly (e.g. for "Acme was founded by Jane", the triple is [Jane, founded, Acme], not the reverse). Also copy the subject's and object's surface text into "subject_text" / "object_text" (exactly as they appear in the ENTITIES list) so the direction is human-checkable.
3. Only extract a relation that is explicitly stated or unambiguously entailed by the TEXT. Do not use outside knowledge, and do not guess relations that the TEXT does not support. If nothing connects two entities in the text, emit nothing for that pair.
4. Choose a concise relation_phrase — prefer a short verb or verb phrase in a canonical present-tense form (e.g. "founded", "born in", "works for", "located in", "acquired", "member of"). Keep phrasing consistent: if two relations are the same kind of relation, give them the exact same relation_phrase (same spelling, same casing).
5. Choose the most specific relation_phrase the evidence supports, but general enough to be reusable across documents (a stable, commonly-recognized relation), not a one-off phrase that fits only this sentence. Stay grounded in the text: do not over-generalize, and do not assert a relation the evidence does not support.
6. "evidence" is the shortest verbatim span from the TEXT that states the relation — copied exactly, character-for-character, from the TEXT. "confidence" is your certainty in the triple (subject, relation_phrase, object, and direction), a value between 0 and 1; lower it when the relation is implicit, ambiguous, or the direction is uncertain.
7. The same pair of entities may hold more than one relation (emit one triple per distinct relation). The same relation_phrase may connect many different pairs. Do not emit the exact same [subject_id, relation_phrase, object_id] triple more than once.
8. After extracting all relations, collect every unique relation_phrase you used. For each, provide: (a) "definition": a concise one-sentence definition of the relation in general terms, phrased so the direction is explicit (what the subject is to the object); and (b) "synonyms": a short list of 2-5 alternative phrasings or near-synonyms of the relation. Both must be general and independent of the specific entities in the text.
9. The set of distinct relation_phrase values used in "relations" must be exactly the set defined in "relation_types": every phrase used has exactly one definition, and every defined phrase is used by at least one relation.
10. Return ONLY valid JSON matching the schema below. No explanations, no extra text.
OUTPUT SCHEMA:
{
  "relations": [
    { "subject_id": "E1", "subject_text": "subject surface text", "relation_phrase": "relation label", "object_id": "E2", "object_text": "object surface text", "evidence": "verbatim span from TEXT", "confidence": <0-1> }
  ],
  "relation_types": [
    { "relation_phrase": "relation label", "definition": "A concise general definition, stating what the subject is to the object.", "synonyms": ["alternative phrasing", "near-synonym"] }
  ]
}"""

def _entities_block(cards):
    return "\n".join(
        f'  {c["id"]} | text: "{c["text"]}" | type: {c["type"]}'
        + (f' | aka: {", ".join(c["aliases"])}' if c.get("aliases") else "")
        for c in cards)

def extract_relations(aid, write=True, verbose=True):
    """Run Gemini open relation extraction over the article's entities -> results/<id>/oie_relations.json."""
    text = (DATASET_DIR / f"{aid}.txt").read_text()
    ref_date = _reference_date(text)
    fname, cards = load_entities(aid)
    ids = {c["id"] for c in cards}
    by_id = {c["id"]: c for c in cards}
    if verbose:
        print(f"[{aid}] ~{len(text)//4} tokens; {len(cards)} entities; REFERENCE_DATE={ref_date}; calling {GEMINI_MODEL} ...")
    user_message = (
        (f"REFERENCE_DATE: {ref_date}\n\n" if ref_date else "")
        + f"ENTITIES:\n{_entities_block(cards)}\n\nTEXT:\n{text}")
    oie = gemini_json(REL_OIE_SYSTEM_PROMPT, user_message)

    # Keep only relations whose subject and object are both real entity ids and are distinct.
    clean, dropped, seen = [], 0, set()
    for r in oie.get("relations", []):
        s, o = r.get("subject_id"), r.get("object_id")
        key = (s, r.get("relation_phrase"), o)
        if s in ids and o in ids and s != o and key not in seen:
            seen.add(key)
            r["subject_text"] = by_id[s]["text"]        # trust the id, re-stamp the surface text
            r["object_text"]  = by_id[o]["text"]
            clean.append(r)
        else:
            dropped += 1
    # Keep only relation_types actually used (and warn about any used phrase left undefined).
    used = list(dict.fromkeys(r["relation_phrase"] for r in clean))
    rtypes = {t["relation_phrase"]: t for t in oie.get("relation_types", []) if t.get("relation_phrase") in used}
    for ph in used:
        rtypes.setdefault(ph, {"relation_phrase": ph, "definition": "", "synonyms": []})

    out = {"article_id": aid, "entities_file": fname,
           "entities": cards, "relations": clean,
           "relation_types": [rtypes[ph] for ph in used]}
    if write:
        out_dir = RESULTS_DIR / aid
        out_dir.mkdir(parents=True, exist_ok=True)
        (out_dir / "oie_relations.json").write_text(json.dumps(out, indent=2, ensure_ascii=False))
    if verbose:
        note = f" ({dropped} dropped: bad/duplicate ids)" if dropped else ""
        print(f"  latency {last_latency_s:.2f}s | {len(clean)} relations{note}, "
              f"{len(used)} distinct relation phrases -> results/{aid}/oie_relations.json")
    return out

# oie = extract_relations(ARTICLE_ID)

## 3. Inspect the demo article

In [10]:
# Extract relations for the single demo article and print them as readable triples.
oie = extract_relations(ARTICLE_ID)
print("-" * 74)
for r in oie["relations"]:
    print(f'  {r["subject_text"]!r}  --{r["relation_phrase"]}-->  {r["object_text"]!r}   (conf {r.get("confidence")})')
    print(f'      ev: "{r.get("evidence","")}"')
print("-" * 74)
print("relation phrases:")
for t in oie["relation_types"]:
    print(f'  {t["relation_phrase"]:<22} {t.get("definition","")}')

[art1] ~1170 tokens; 103 entities; REFERENCE_DATE=2026-07-03; calling gemini-3.6-flash ...
  latency 31.04s | 15 relations, 12 distinct relation phrases -> results/art1/oie_relations.json
--------------------------------------------------------------------------
  'France'  --recorded-->  'more than 2,000 deaths'   (conf 0.95)
      ev: "France has said it recorded an increase of more than 2,000 deaths"
  'Belgium'  --recorded-->  '1,222 excess deaths'   (conf 0.95)
      ev: "Belgium recorded 1,222 excess deaths"
  'French health minister Stéphanie Rist'  --government official of-->  'France'   (conf 0.95)
      ev: "French health minister Stéphanie Rist"
  'French Interior Minister Laurent Nuñez'  --government official of-->  'France'   (conf 0.95)
      ev: "French Interior Minister Laurent Nuñez"
  "Portugal's government"  --government of-->  'Portugal'   (conf 0.95)
      ev: "Portugal's government"
  "Portugal's government"  --declared-->  'a state of alert'   (conf 0.95)
      e

## 4. Batch — process the selected articles (`ARTICLE_IDS`)

In [ ]:
# Runs relation extraction for each article in ARTICLE_IDS (choose them via SELECTED_ARTICLES in the
# config cell). Re-runs overwrite existing outputs. (Property canonicalization comes next, in nb7.)
for aid in ARTICLE_IDS:
    print("=" * 60, aid)
    extract_relations(aid)
print(f"\ndone — extracted relations for {len(ARTICLE_IDS)} article(s)")

============================================================ art1
[art1] ~1170 tokens; 103 entities; REFERENCE_DATE=2026-07-03; calling gemini-3.6-flash ...
  latency 26.42s | 12 relations, 8 distinct relation phrases -> results/art1/oie_relations.json
============================================================ art2
[art2] ~1865 tokens; 91 entities; REFERENCE_DATE=2026-07-03; calling gemini-3.6-flash ...
  latency 23.66s | 14 relations, 11 distinct relation phrases -> results/art2/oie_relations.json
============================================================ art4
[art4] ~666 tokens; 76 entities; REFERENCE_DATE=2025-09-03; calling gemini-3.6-flash ...
  latency 33.07s | 20 relations, 5 distinct relation phrases -> results/art4/oie_relations.json
============================================================ art5
[art5] ~1147 tokens; 89 entities; REFERENCE_DATE=2026-07-03; calling gemini-3.6-flash ...


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [17]:
extract_relations("art5")

[art5] ~1147 tokens; 89 entities; REFERENCE_DATE=2026-07-03; calling gemini-2.5-flash ...
  latency 121.09s | 93 relations, 51 distinct relation phrases -> results/art5/oie_relations.json


{'article_id': 'art5',
 'entities_file': 'typed_deduped_entities_ext_consolidated.json',
 'entities': [{'id': 'E1',
   'text': 'Ebola treatments trial',
   'type': 'MedicalTrial',
   'aliases': []},
  {'id': 'E2',
   'text': 'Democratic Republic of Congo',
   'type': 'Country',
   'aliases': ['DR Congo']},
  {'id': 'E3', 'text': 'Belgium', 'type': 'Country', 'aliases': []},
  {'id': 'E4', 'text': 'UK', 'type': 'Country', 'aliases': []},
  {'id': 'E5', 'text': 'Uganda', 'type': 'Country', 'aliases': []},
  {'id': 'E6', 'text': 'France', 'type': 'Country', 'aliases': []},
  {'id': 'E7', 'text': 'Cape Verde', 'type': 'Country', 'aliases': []},
  {'id': 'E8', 'text': 'Argentina', 'type': 'Country', 'aliases': []},
  {'id': 'E9', 'text': 'India', 'type': 'Country', 'aliases': []},
  {'id': 'E10',
   'text': 'World Health Organization (WHO)',
   'type': 'InternationalOrganization',
   'aliases': ['WHO']},
  {'id': 'E11',
   'text': 'Tedros Adhanom Ghebreyesus',
   'type': 'Person',
   'alias